In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
import eda_helper_functions
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.ensemble import IsolationForest
import sklearn

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
    PowerTransformer,
    FunctionTransformer
)

from feature_engine.outliers import Winsorizer
from feature_engine.datetime import DatetimeFeatures
from feature_engine.selection import SelectBySingleFeaturePerformance
from feature_engine.encoding import (
    RareLabelEncoder,
    MeanEncoder,
    CountFrequencyEncoder
)

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Taking the raw data
flights = pd.read_csv("../data/raw/flight_price.csv")
flights.head(5)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [4]:
# Removing the duplicates from the data 
remove_index = flights[flights.duplicated()].index
flights.drop(index = remove_index,inplace = True)

In [5]:
# Descrption of the dataset

# <class 'pandas.core.frame.DataFrame'>
# Index: 10682 entries, 0 to 10682
# Data columns (total 9 columns):
#  #   Column           Non-Null Count  Dtype         
# ---  ------           --------------  -----         
#  0   Airline          10682 non-null  object        
#  1   Date_of_Journey  10682 non-null  datetime64[ns]
#  2   Source           10682 non-null  object        
#  3   Destination      10682 non-null  object        
#  4   Dep_Time         10682 non-null  object        
#  5   Duration         10682 non-null  object        
#  6   Total_Stops      10682 non-null  int64         
#  7   Additional_Info  10682 non-null  object        
#  8   Price            10682 non-null  int64         
# dtypes: datetime64[ns](1), int64(2), object(6)
# memory usage: 834.5+ KB

In [6]:
# Function to change to the numercical stop

def change_to_numerical_stops(value):
    if(value == 'non-stop'):
        return 0
    elif (value == '1 stop'):
        return 1
    elif (value == '2 stops'):
        return 2
    elif (value == '3 stops'):
        return 3
    else:
        return 4

In [7]:
# Converting the duration to minutes

def change_duration_type(value):
    parts = value.split()
    if(len(parts) == 1):
        if 'h' in parts[0]:
            time = int(parts[0].replace('h','')) * 60
            return time
        else:
            time = int(parts[0].replace('m',''))
            return time
    hour = int(parts[0].replace('h',''))
    minutes = int(parts[1].replace('m',''))
    time = hour * 60 + minutes
    return time

In [8]:
# Cleaning the raw Data

def clean_dataframe(df):
    return(
        df
        .assign(**{
            col: df[col].str.lower().str.strip()
            for col in df.select_dtypes(include = "O").columns 

        })
        .rename(columns = str.lower)
        .assign(
            airline = lambda df_:( 
                df_.airline
                .str.replace(" premium economy","")
                .str.replace(" business","")
                .str.title()
            ),
            total_stops = df.Total_Stops.apply(change_to_numerical_stops),
            duration = df.Duration.apply(change_duration_type),
            dep_time = pd.to_datetime(df.Dep_Time,format='%H:%M'),
            dep_time_hour = lambda df_: df_.dep_time.dt.hour,
            dep_time_min = lambda df_: df_.dep_time.dt.minute,
            date_of_journey = pd.to_datetime(df.Date_of_Journey,dayfirst=True,format='%d/%m/%Y'),
            dtoj_day = lambda df_:df_.date_of_journey.dt.day,
            dtoj_month = lambda df_:df_.date_of_journey.dt.month,
            dtoj_year = lambda df_:df_.date_of_journey.dt.year
        )
        .drop(columns = ['date_of_journey','dep_time','route','arrival_time'])
        .drop_duplicates()
    )


In [9]:
import os
flights_cleaned = clean_dataframe(flights)
cleaned_flight_data_path = os.path.join("..", "data","processed","cleaned_flight_data.csv")
flights_cleaned.to_csv(cleaned_flight_data_path, index=False)
print(f"✅ cleaned_flight_data saved as {cleaned_flight_data_path}")

✅ cleaned_flight_data saved as ..\data\processed\cleaned_flight_data.csv


In [10]:
flights_cleaned.head(2)

,airline,source,destination,duration,total_stops,additional_info,price,dep_time_hour,dep_time_min,dtoj_day,dtoj_month,dtoj_year
0,Indigo,banglore,new delhi,170,0,no info,3897,22,20,24,3,2019
1,Air India,kolkata,banglore,445,2,no info,7662,5,50,1,5,2019


In [11]:
df = flights_cleaned.copy()
df["source"] = df["source"].str.lower()
df["destination"] = df["destination"].str.lower()

df["route_key"] = list(
    zip(df["source"], df["destination"])
)

route_counts = df["route_key"].value_counts()

valid_routes = route_counts[route_counts >= 2].index

df = df[
    df["route_key"].isin(valid_routes)
]


In [12]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["price"])
y = df["price"]

X_, X_test, y_, y_test = train_test_split(X,y,test_size=0.2,random_state=0,stratify=df["route_key"])
X_train, X_val, y_train, y_val = train_test_split(X_,y_,test_size=0.2,random_state=0,stratify=X_["route_key"])


In [13]:
print("Train distribution:")
print(X_train["route_key"].value_counts(normalize=True))

print("\nValidation distribution:")
print(X_val["route_key"].value_counts(normalize=True))

print("\nTest distribution:")
print(X_test["route_key"].value_counts(normalize=True))


Train distribution:
route_key
(delhi, cochin)          0.415447
(kolkata, banglore)      0.273379
(banglore, delhi)        0.121004
(banglore, new delhi)    0.087093
(mumbai, hyderabad)      0.066627
(chennai, kolkata)       0.036451
Name: proportion, dtype: float64

Validation distribution:
route_key
(delhi, cochin)          0.415173
(kolkata, banglore)      0.273596
(banglore, delhi)        0.120669
(banglore, new delhi)    0.087216
(mumbai, hyderabad)      0.066906
(chennai, kolkata)       0.036440
Name: proportion, dtype: float64

Test distribution:
route_key
(delhi, cochin)          0.415671
(kolkata, banglore)      0.273292
(banglore, delhi)        0.120879
(banglore, new delhi)    0.087434
(mumbai, hyderabad)      0.066412
(chennai, kolkata)       0.036312
Name: proportion, dtype: float64


In [14]:
train_data = pd.concat([X_train, y_train], axis=1)
val_data = pd.concat([X_val, y_val], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv("../data/processed/train_data.csv", index=False)
val_data.to_csv("../data/processed/val_data.csv", index=False)
test_data.to_csv("../data/processed/test_data.csv", index=False)

print("Saving Succesful")
print(X_train.shape,y_train.shape)
print(X_val.shape,y_val.shape)
print(X_test.shape,y_test.shape)

Saving Succesful
(6694, 12) (6694,)
(1674, 12) (1674,)
(2093, 12) (2093,)
